In [1]:
import pandas as pd

base_path = "/content/"

orders = pd.read_csv(base_path + "olist_orders_dataset.csv")
order_items = pd.read_csv(base_path + "olist_order_items_dataset.csv")
reviews = pd.read_csv(base_path + "olist_order_reviews_dataset.csv")
products = pd.read_csv(base_path + "olist_products_dataset.csv")

print(len(orders), len(order_items), len(reviews), len(products))

99441 112650 99224 32951


In [2]:
!pip -q install great_expectations==0.17.21

In [3]:
import great_expectations as ge

# Converte para GE DataFrames
ge_orders = ge.from_pandas(orders)
ge_items = ge.from_pandas(order_items)
ge_reviews = ge.from_pandas(reviews)
ge_products = ge.from_pandas(products)

results = {}

  return datetime.utcnow().replace(tzinfo=utc)



In [4]:
# -----------------------
# ORDERS
# -----------------------
orders_suite = "orders_suite"
ge_orders.expect_column_values_to_not_be_null("order_id")
ge_orders.expect_column_values_to_be_unique("order_id")
ge_orders.expect_column_values_to_not_be_null("order_purchase_timestamp")
ge_orders.expect_column_values_to_be_in_set(
    "order_status",
    ["delivered", "shipped", "canceled", "invoiced", "processing", "unavailable", "approved", "created"]
)

results["orders"] = ge_orders.validate()

In [5]:
# -----------------------
# ORDER_ITEMS
# -----------------------
items_suite = "order_items_suite"
ge_items.expect_column_values_to_not_be_null("order_id")
ge_items.expect_column_values_to_not_be_null("product_id")
ge_items.expect_column_values_to_not_be_null("order_item_id")
ge_items.expect_column_values_to_be_between("price", min_value=0)
ge_items.expect_column_values_to_be_between("freight_value", min_value=0)

results["order_items"] = ge_items.validate()


In [6]:
# -----------------------
# REVIEWS
# -----------------------
reviews_suite = "reviews_suite"
ge_reviews.expect_column_values_to_not_be_null("review_id")
ge_reviews.expect_column_values_to_be_between("review_score", min_value=1, max_value=5)

results["reviews"] = ge_reviews.validate()

In [7]:
# -----------------------
# PRODUCTS
# -----------------------
products_suite = "products_suite"
ge_products.expect_column_values_to_not_be_null("product_id")
ge_products.expect_column_values_to_be_unique("product_id")

# Se existirem colunas numéricas, validar >= 0
numeric_cols = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]
for c in numeric_cols:
    if c in ge_products.columns:
        ge_products.expect_column_values_to_be_between(c, min_value=0)

results["products"] = ge_products.validate()

# Mostrar resumo simples de sucesso/fracasso
summary = {}
for k,v in results.items():
    summary[k] = {
        "success": v["success"],
        "evaluated_expectations": v["statistics"]["evaluated_expectations"],
        "successful_expectations": v["statistics"]["successful_expectations"],
        "unsuccessful_expectations": v["statistics"]["unsuccessful_expectations"],
    }

summary

{'orders': {'success': True,
  'evaluated_expectations': 4,
  'successful_expectations': 4,
  'unsuccessful_expectations': 0},
 'order_items': {'success': True,
  'evaluated_expectations': 5,
  'successful_expectations': 5,
  'unsuccessful_expectations': 0},
 'reviews': {'success': True,
  'evaluated_expectations': 2,
  'successful_expectations': 2,
  'unsuccessful_expectations': 0},
 'products': {'success': True,
  'evaluated_expectations': 6,
  'successful_expectations': 6,
  'unsuccessful_expectations': 0}}

In [10]:
import os, json

os.makedirs("/content/dq_reports", exist_ok=True)

for name, res in results.items():
    # converte o ValidationResult para um dict serializável
    res_dict = res.to_json_dict()
    with open(f"/content/dq_reports/{name}_validation.json", "w", encoding="utf-8") as f:
        json.dump(res_dict, f, ensure_ascii=False, indent=2)

print("JSONs gerados em: /content/dq_reports")

JSONs gerados em: /content/dq_reports


In [11]:
def null_rate(df, cols):
    return (df[cols].isna().mean().sort_values(ascending=False) * 100).round(2)

print("Null rates - orders:")
print(null_rate(orders, ["order_id", "order_purchase_timestamp", "order_status"]))

print("\nNull rates - order_items:")
print(null_rate(order_items, ["order_id", "order_item_id", "product_id", "price", "freight_value"]))

print("\nNull rates - reviews:")
print(null_rate(reviews, ["review_id", "review_score", "review_comment_message"]))

print("\nNull rates - products:")
print(null_rate(products, ["product_id", "product_category_name"]))


  return datetime.utcnow().replace(tzinfo=utc)



Null rates - orders:
order_id                    0.0
order_purchase_timestamp    0.0
order_status                0.0
dtype: float64

Null rates - order_items:
order_id         0.0
order_item_id    0.0
product_id       0.0
price            0.0
freight_value    0.0
dtype: float64

Null rates - reviews:
review_comment_message    58.7
review_id                  0.0
review_score               0.0
dtype: float64

Null rates - products:
product_category_name    1.85
product_id               0.00
dtype: float64


In [12]:
print("Invalids - order_items (price < 0):", (order_items["price"] < 0).sum())
print("Invalids - order_items (freight_value < 0):", (order_items["freight_value"] < 0).sum())

print("Invalids - reviews (review_score not in 1..5):",
      (~reviews["review_score"].between(1, 5)).sum())

Invalids - order_items (price < 0): 0
Invalids - order_items (freight_value < 0): 0
Invalids - reviews (review_score not in 1..5): 0
